# Uber Ride Analytics - Python EDA

Dataset: `uber_data_analytics_clean.csv` (150,000 bookings, Delhi-NCR)

This notebook covers:
1. Load & type-correct the data
2. Parse Date + Time into a proper datetime column
3. Cross-check derived hour/weekend logic against SQL results
4. Hour x day-of-week demand pivot (heatmap data)
5. Fare-vs-distance correlation (+ full numeric correlation matrix)
6. Zone-wise demand summary (groupby)
7. Surge-time pattern check (volume vs. wait time vs. cancellation rate)


## 1. Load and type-correct the data

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('uber_data_analytics_clean.csv')

print(df.dtypes)
print(df.shape)

Numeric columns (`Booking Value`, `Ride Distance`, `Avg VTAT`, `Avg CTAT`, `Driver Ratings`, `Customer Rating`) import correctly as `float64` in pandas. Only `Date` and `Time` need explicit parsing (unlike SQLiteViz, where numeric columns imported as text).

## 2. Parse Date + Time into a proper datetime column

In [ ]:
df['pickup_datetime'] = pd.to_datetime(
    df['Date'] + ' ' + df['Time'],
    format='%m/%d/%Y %I:%M:%S %p'
)

df['hour_24'] = df['pickup_datetime'].dt.hour
df['day_of_week'] = df['pickup_datetime'].dt.day_name()
df['is_weekend'] = df['pickup_datetime'].dt.dayofweek >= 5

print(df[['Date', 'Time', 'pickup_datetime', 'hour_24', 'day_of_week', 'is_weekend']].head(10))

`%m/%d/%Y` matches the `M/D/YYYY` date format; `%I:%M:%S %p` matches the 12-hour AM/PM time format. `dayofweek >= 5` flags Saturday/Sunday as weekend - same definition used in the SQL `strftime('%w', Date) IN ('0','6')` logic, so Python and SQL weekend definitions match.

## 3. Cross-check against SQL results

In [ ]:
# Should match Query 1's time-of-day buckets and Query 3's peak hour
hour_counts = df['hour_24'].value_counts().sort_index()
print(hour_counts)

print("\nPeak hour:", df['hour_24'].value_counts().idxmax())

weekend_check = df.groupby('is_weekend').size()
print("\n", weekend_check)

**Confirmed**: peak hour = 18 (6 PM), matching SQL Query 3. Weekend count (~42,940) vs weekday (~107,060) is close to the expected 2/7 ratio.

## 4. Hour x day-of-week demand pivot (heatmap data)

In [ ]:
heatmap_data = df.pivot_table(
    index='day_of_week',
    columns='hour_24',
    values='Booking ID',
    aggfunc='count',
    fill_value=0
)

day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
heatmap_data = heatmap_data.reindex(day_order)

print(heatmap_data)

# Save for Power BI / Excel
heatmap_data.to_csv('heatmap_hour_day.csv')

**Finding**: booking counts at each hour are nearly identical across all seven days (e.g. hour 18: Monday 1835, Saturday 1800, Sunday 1756) - no meaningful weekday-vs-weekend shape difference. Demand appears to be hour-driven, not day-of-week-driven, in this dataset.

## 5. Fare-vs-distance correlation (+ full correlation matrix)

In [ ]:
# Filter to completed rides only - Booking Value/Ride Distance are null otherwise
completed = df[df['Booking Status'] == 'Completed'].copy()

# Core correlation: fare vs distance
fare_distance_corr = completed['Booking Value'].corr(completed['Ride Distance'])
print("Fare vs Distance correlation:", round(fare_distance_corr, 4))

# Broader check: correlation matrix across all relevant numeric columns
numeric_cols = ['Booking Value', 'Ride Distance', 'Avg VTAT', 'Avg CTAT',
                 'Driver Ratings', 'Customer Rating']
corr_matrix = completed[numeric_cols].corr()
print("\nFull correlation matrix:")
print(corr_matrix.round(3))

**Finding**: every off-diagonal value in the correlation matrix is essentially zero (all under 0.01 in magnitude). Fare, distance, VTAT, CTAT, driver ratings, and customer ratings show no meaningful linear relationship with each other - strong evidence these fields were generated independently rather than derived from a real fare/trip formula.

## 6. Zone-wise demand summary (groupby)

In [ ]:
zone_summary = completed.groupby('Pickup Location').agg(
    trip_count=('Booking ID', 'count'),
    avg_fare=('Booking Value', 'mean'),
    avg_distance=('Ride Distance', 'mean'),
    avg_driver_rating=('Driver Ratings', 'mean')
).round(2).sort_values('trip_count', ascending=False)

print(zone_summary.head(20))
print("\nTotal distinct zones:", zone_summary.shape[0])

zone_summary.to_csv('zone_summary.csv')

**Finding**: across 176 zones, trip count, avg fare, avg distance, and avg driver rating are all tightly clustered with no standout zone - demand and fare are evenly distributed rather than concentrated in specific areas.

## 7. Surge-time pattern check

This dataset has no explicit surge-multiplier column, so "surge-time" is checked via proxy: does booking volume correlate with degraded service (higher wait time / higher cancellation rate) at the same hours?

In [ ]:
hourly_pattern = df.groupby('hour_24').agg(
    total_bookings=('Booking ID', 'count'),
    avg_vtat=('Avg VTAT', 'mean'),
    cancelled=('Booking Status', lambda x: x.isin(['Cancelled by Driver', 'Cancelled by Customer']).sum())
).round(2)

hourly_pattern['cancellation_rate_pct'] = (
    hourly_pattern['cancelled'] / hourly_pattern['total_bookings'] * 100
).round(2)

print(hourly_pattern.sort_values('total_bookings', ascending=False))

**Finding**: `avg_vtat` stays in a tight 8.36-8.55 minute band regardless of hour, and `cancellation_rate_pct` stays in a 22-26% band regardless of volume. No surge-time signal - wait time and cancellation rate are effectively independent of booking volume across all 24 hours.

## Summary

Across fare, distance, zone, day-of-week, and hourly demand, this dataset's operational metrics (VTAT, cancellation rate, fare) show no meaningful correlation with volume or trip characteristics - indicating the numeric fields were likely generated independently rather than simulating realistic surge/demand-responsive behavior. The dataset remains valuable for demonstrating SQL/pandas/BI workflow skills, but shouldn't be over-interpreted as reflecting real-world Uber demand dynamics.